# RRT & RRT* Path Planning
## Sampling-Based Motion Planning for Robotics

### Learning Objectives

- Understand the **configuration space** abstraction and why it matters for planning
- Implement the **Rapidly-exploring Random Tree (RRT)** algorithm from scratch
- Understand **probabilistic completeness** and verify it experimentally
- Implement **RRT*** with rewiring for asymptotically optimal planning
- Compare RRT vs RRT* in terms of path quality and computational cost
- Apply sampling-based planning to **multi-link robot arms** in C-space

### Prerequisites

- Basic probability theory (uniform distributions, convergence)
- Euclidean geometry (distances, line-segment intersections)
- Linear algebra (rotation matrices, forward kinematics)
- Python proficiency (NumPy, Matplotlib)

### Why Sampling-Based Planning?

Grid-based planners (A*, Dijkstra) discretize the search space into cells. For a robot with $d$ degrees of freedom and $k$ discretization levels per dimension, the grid has $k^d$ cells. A 6-DOF robot arm with 360 angular bins per joint yields $360^6 \approx 2.2 \times 10^{15}$ cells — far too many to enumerate. Sampling-based planners like RRT avoid this **curse of dimensionality** by building a search tree incrementally through random sampling.

### References

- LaValle, S.M. *Planning Algorithms*, Cambridge University Press, 2006
- Karaman, S. & Frazzoli, E. "Sampling-based algorithms for optimal motion planning," *IJRR*, 2011
- LaValle, S.M. "Rapidly-exploring random trees: A new tool for path planning," 1998
- Gammell, J.D. et al. "Informed RRT*: Optimal sampling-based path planning," *IROS*, 2014

---
## 2. Configuration Space

### Workspace vs Configuration Space

The **workspace** $\mathcal{W} \subseteq \mathbb{R}^2$ (or $\mathbb{R}^3$) is the physical space where the robot operates. The **configuration space** (C-space) $\mathcal{C}$ is the space of all possible robot configurations $q$. For a planar robot arm with $n$ revolute joints:

$$\mathcal{C} = [0, 2\pi)^n$$

Each point $q = (\theta_1, \theta_2, \ldots, \theta_n) \in \mathcal{C}$ maps to a specific robot pose in the workspace through **forward kinematics**.

### Obstacle Mapping

Workspace obstacles $\mathcal{O}_W$ induce C-space obstacles $\mathcal{O}_C$:

$$\mathcal{O}_C = \{q \in \mathcal{C} \mid A(q) \cap \mathcal{O}_W \neq \emptyset\}$$

where $A(q)$ is the set of workspace points occupied by the robot at configuration $q$. The **free configuration space** is:

$$\boxed{\mathcal{C}_{\text{free}} = \mathcal{C} \setminus \mathcal{O}_C}$$

Planning reduces to finding a continuous path in $\mathcal{C}_{\text{free}}$ from $q_{\text{start}}$ to $q_{\text{goal}}$.

### 2R Planar Arm

For a 2-link planar arm with link lengths $L_1$ and $L_2$, the forward kinematics are:

| Quantity | Expression |
|----------|------------|
| Elbow position | $(L_1 \cos\theta_1,\; L_1 \sin\theta_1)$ |
| End-effector position | $(L_1 \cos\theta_1 + L_2 \cos(\theta_1 + \theta_2),\; L_1 \sin\theta_1 + L_2 \sin(\theta_1 + \theta_2))$ |

A line segment (robot link) collides with a circular obstacle of center $c$ and radius $r$ if the minimum distance from the segment to $c$ is less than $r$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch
from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors
from scipy.spatial import KDTree
import time

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2
})

np.random.seed(42)
print("Setup complete.")

In [ ]:
# ================================================================
# Global Constants
# ================================================================

# 2R arm parameters
L1 = 1.0          # Length of link 1
L2 = 0.8          # Length of link 2

# 3R arm parameters
L1_3R = 1.0       # Length of link 1 (3R arm)
L2_3R = 0.8       # Length of link 2 (3R arm)
L3_3R = 0.5       # Length of link 3 (3R arm)

# 2D workspace bounds
WORKSPACE_BOUNDS = [0.0, 10.0, 0.0, 10.0]  # [x_min, x_max, y_min, y_max]

# RRT parameters
STEP_SIZE = 0.5
GOAL_BIAS = 0.05
MAX_ITER = 3000
GOAL_THRESHOLD = 0.5

# RRT* parameters
GAMMA_RRT_STAR = 50.0
GAMMA_3D = 80.0

# Collision checking
N_COLLISION_SAMPLES = 20  # Points sampled along segment for collision check

# C-space resolution
CSPACE_RESOLUTION = 200  # Grid resolution for C-space visualization

# Color palette
COLOR_STEELBLUE = 'steelblue'
COLOR_CORAL = 'coral'
COLOR_SEAGREEN = 'seagreen'
COLOR_GOLDENROD = 'goldenrod'

print("Constants loaded.")

In [ ]:
def arm_forward_kinematics(theta1, theta2, l1=L1, l2=L2):
    """Compute joint positions for a 2R planar arm.

    The base is at the origin. Returns the base, elbow, and end-effector
    positions for the given joint angles.

    Args:
        theta1: Angle of joint 1 (radians)
        theta2: Angle of joint 2 (radians), relative to link 1
        l1: Length of link 1
        l2: Length of link 2

    Returns:
        positions: Array of shape (3, 2) with [base, elbow, end-effector] positions
    """
    base = np.array([0.0, 0.0])
    elbow = np.array([l1 * np.cos(theta1), l1 * np.sin(theta1)])
    end_effector = elbow + np.array([
        l2 * np.cos(theta1 + theta2),
        l2 * np.sin(theta1 + theta2)
    ])
    return np.array([base, elbow, end_effector])


def point_to_segment_distance(point, seg_start, seg_end):
    """Compute minimum distance from a point to a line segment.

    Args:
        point: Shape (2,) — the query point
        seg_start: Shape (2,) — start of segment
        seg_end: Shape (2,) — end of segment

    Returns:
        dist: Minimum distance from point to the segment
    """
    v = seg_end - seg_start
    u = point - seg_start
    t = np.dot(u, v) / (np.dot(v, v) + 1e-12)
    t = np.clip(t, 0.0, 1.0)
    closest = seg_start + t * v
    return np.linalg.norm(point - closest)


def arm_in_collision(theta1, theta2, obstacles, l1=L1, l2=L2):
    """Check if a 2R arm configuration collides with circular obstacles.

    Tests both links of the arm against all obstacles. A link collides
    with a circular obstacle if the minimum distance from the link
    segment to the obstacle center is less than the obstacle radius.

    Args:
        theta1: Angle of joint 1 (radians)
        theta2: Angle of joint 2 (radians)
        obstacles: List of dicts with keys 'center' (tuple) and 'radius' (float)
        l1: Length of link 1
        l2: Length of link 2

    Returns:
        collision: True if the arm collides with any obstacle
    """
    positions = arm_forward_kinematics(theta1, theta2, l1, l2)
    base, elbow, end_eff = positions

    for obs in obstacles:
        center = np.array(obs['center'])
        radius = obs['radius']
        # Check link 1: base -> elbow
        if point_to_segment_distance(center, base, elbow) < radius:
            return True
        # Check link 2: elbow -> end-effector
        if point_to_segment_distance(center, elbow, end_eff) < radius:
            return True
    return False


# Define workspace obstacles for the 2R arm
arm_obstacles = [
    {'center': (1.0, 0.8), 'radius': 0.25},
    {'center': (0.5, 1.2), 'radius': 0.2},
    {'center': (-0.3, 1.0), 'radius': 0.3},
    {'center': (0.8, -0.5), 'radius': 0.2},
]

# Grid-sample C-space
theta1_range = np.linspace(0, 2 * np.pi, CSPACE_RESOLUTION)
theta2_range = np.linspace(0, 2 * np.pi, CSPACE_RESOLUTION)
cspace_map = np.zeros((CSPACE_RESOLUTION, CSPACE_RESOLUTION), dtype=bool)

for i, t1 in enumerate(theta1_range):
    for j, t2 in enumerate(theta2_range):
        cspace_map[j, i] = arm_in_collision(t1, t2, arm_obstacles)

print(f"C-space computed: {CSPACE_RESOLUTION}x{CSPACE_RESOLUTION} grid")
print(f"Obstacle fraction: {cspace_map.mean():.3f}")
print(f"Free fraction: {1 - cspace_map.mean():.3f}")

In [ ]:
# Visualize workspace and C-space side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: Workspace with obstacles and sample arm configurations ---
ax = axes[0]
ax.set_title('Workspace: 2R Arm and Obstacles', fontsize=14, fontweight='bold')
ax.set_xlim(-2.0, 2.0)
ax.set_ylim(-2.0, 2.0)
ax.set_aspect('equal')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')

# Draw obstacles
for obs in arm_obstacles:
    circle = Circle(obs['center'], obs['radius'], color=COLOR_CORAL, alpha=0.6)
    ax.add_patch(circle)

# Draw workspace boundary (reachable region)
theta_vis = np.linspace(0, 2 * np.pi, 200)
ax.plot((L1 + L2) * np.cos(theta_vis), (L1 + L2) * np.sin(theta_vis),
        '--', color='gray', alpha=0.4, label='Reach boundary')
ax.plot(abs(L1 - L2) * np.cos(theta_vis), abs(L1 - L2) * np.sin(theta_vis),
        '--', color='gray', alpha=0.4)

# Draw a few sample arm configurations
sample_configs = [(0.5, 1.0), (1.5, 2.5), (2.8, 0.5), (4.0, 3.0)]
colors_arm = [COLOR_STEELBLUE, COLOR_SEAGREEN, COLOR_GOLDENROD, 'purple']
for (t1, t2), c in zip(sample_configs, colors_arm):
    positions = arm_forward_kinematics(t1, t2)
    ax.plot(positions[:, 0], positions[:, 1], 'o-', color=c, linewidth=3,
            markersize=6, alpha=0.8)

ax.legend(loc='lower left', fontsize=10)

# --- Right: C-space obstacle map ---
ax = axes[1]
ax.set_title('Configuration Space Obstacle Map', fontsize=14, fontweight='bold')
ax.imshow(cspace_map, extent=[0, 2*np.pi, 0, 2*np.pi], origin='lower',
          cmap='RdBu_r', alpha=0.8, aspect='equal')
ax.set_xlabel(r'$\theta_1$ (rad)')
ax.set_ylabel(r'$\theta_2$ (rad)')
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels(['0', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$'])
ax.set_yticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_yticklabels(['0', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$'])

# Mark sample configurations in C-space
for (t1, t2), c in zip(sample_configs, colors_arm):
    ax.plot(t1, t2, 'o', color=c, markersize=10, markeredgecolor='black',
            markeredgewidth=1.5, zorder=5)

# Add colorbar-like legend
ax.text(0.3, 5.8, 'Red = Collision', fontsize=10, color='red', fontweight='bold')
ax.text(0.3, 5.4, 'Blue = Free', fontsize=10, color='blue', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 3. RRT Algorithm

The **Rapidly-exploring Random Tree (RRT)** builds a tree rooted at $q_{\text{start}}$ by repeatedly:

1. **Sample** a random point $q_{\text{rand}}$ in $\mathcal{C}$
2. **Find** the nearest node $q_{\text{near}}$ in the tree
3. **Steer** from $q_{\text{near}}$ toward $q_{\text{rand}}$ by step size $\eta$ to get $q_{\text{new}}$
4. **Check** if the edge $(q_{\text{near}}, q_{\text{new}})$ is collision-free
5. **Add** $q_{\text{new}}$ to the tree if collision-free

### Pseudocode

```
RRT(q_start, q_goal, max_iter, eta, goal_bias):
    T.init(q_start)
    for i = 1 to max_iter:
        if random() < goal_bias:
            q_rand = q_goal
        else:
            q_rand = SampleRandom(C)
        q_near = NearestNeighbor(T, q_rand)
        q_new = Steer(q_near, q_rand, eta)
        if CollisionFree(q_near, q_new):
            T.add_node(q_new)
            T.add_edge(q_near, q_new)
            if ||q_new - q_goal|| < threshold:
                return ExtractPath(T, q_new)
    return FAILURE
```

### Key Properties

| Property | Value |
|----------|-------|
| Completeness | Probabilistically complete |
| Optimality | Not optimal (finds *a* path, not *the best*) |
| Time complexity | $O(n \log n)$ with KD-tree for nearest neighbor |
| Space complexity | $O(n)$ for tree storage |

The **goal bias** parameter $p_{\text{goal}}$ causes the tree to grow toward the goal with probability $p_{\text{goal}}$, speeding up convergence without sacrificing probabilistic completeness.

In [ ]:
def sample_random(bounds):
    """Sample a uniform random point within the given bounds.

    Args:
        bounds: List [x_min, x_max, y_min, y_max] defining the sampling region

    Returns:
        point: Shape (2,) — random point in the bounded region
    """
    x = np.random.uniform(bounds[0], bounds[1])
    y = np.random.uniform(bounds[2], bounds[3])
    return np.array([x, y])


def nearest_neighbor(tree_nodes, point):
    """Find the index of the nearest node in the tree to the given point.

    Uses Euclidean distance for comparison.

    Args:
        tree_nodes: Shape (N, 2) — array of node positions in the tree
        point: Shape (2,) — query point

    Returns:
        idx: Integer index of the nearest node in tree_nodes
    """
    dists = np.linalg.norm(tree_nodes - point, axis=1)
    return int(np.argmin(dists))


def steer(from_node, to_node, step_size):
    """Move from from_node toward to_node by at most step_size.

    If the distance between nodes is less than step_size, returns to_node.

    Args:
        from_node: Shape (2,) — starting position
        to_node: Shape (2,) — target position
        step_size: Maximum distance to travel

    Returns:
        new_node: Shape (2,) — the new node position
    """
    direction = to_node - from_node
    dist = np.linalg.norm(direction)
    if dist < step_size:
        return to_node.copy()
    return from_node + (direction / dist) * step_size


def collision_free_circles(p1, p2, obstacles):
    """Check if a line segment is collision-free against circular obstacles.

    Samples N_COLLISION_SAMPLES points along the segment and checks each
    against all circular obstacles.

    Args:
        p1: Shape (2,) — start of segment
        p2: Shape (2,) — end of segment
        obstacles: List of dicts, each with 'center' and 'radius' keys

    Returns:
        free: True if the segment does not intersect any obstacle
    """
    for t in np.linspace(0, 1, N_COLLISION_SAMPLES):
        pt = p1 + t * (p2 - p1)
        for obs in obstacles:
            center = np.array(obs['center'])
            if np.linalg.norm(pt - center) < obs['radius']:
                return False
    return True


def collision_free(p1, p2, obstacles):
    """Check if a line segment is collision-free against mixed obstacles.

    Supports both circular obstacles (with 'center'/'radius' keys) and
    rectangular obstacles (with 'corner'/'width'/'height' keys).

    Args:
        p1: Shape (2,) — start of segment
        p2: Shape (2,) — end of segment
        obstacles: List of obstacle dicts

    Returns:
        free: True if the segment does not intersect any obstacle
    """
    for t in np.linspace(0, 1, N_COLLISION_SAMPLES):
        pt = p1 + t * (p2 - p1)
        for obs in obstacles:
            if 'center' in obs:
                center = np.array(obs['center'])
                if np.linalg.norm(pt - center) < obs['radius']:
                    return False
            elif 'corner' in obs:
                cx, cy = obs['corner']
                w, h = obs['width'], obs['height']
                if cx <= pt[0] <= cx + w and cy <= pt[1] <= cy + h:
                    return False
    return True


def point_in_obstacle(point, obstacles):
    """Check if a single point lies inside any obstacle.

    Args:
        point: Shape (2,) — the point to check
        obstacles: List of obstacle dicts

    Returns:
        inside: True if the point is inside any obstacle
    """
    for obs in obstacles:
        if 'center' in obs:
            if np.linalg.norm(point - np.array(obs['center'])) < obs['radius']:
                return True
        elif 'corner' in obs:
            cx, cy = obs['corner']
            w, h = obs['width'], obs['height']
            if cx <= point[0] <= cx + w and cy <= point[1] <= cy + h:
                return True
    return False


def build_rrt(start, goal, obstacles, bounds, max_iter=MAX_ITER,
              step_size=STEP_SIZE, goal_bias=GOAL_BIAS,
              goal_threshold=GOAL_THRESHOLD):
    """Build a Rapidly-exploring Random Tree from start toward goal.

    The tree is represented as a list of nodes and a corresponding list of
    parent indices. Node 0 is the start (parent = -1).

    Args:
        start: Shape (2,) — start position
        goal: Shape (2,) — goal position
        obstacles: List of obstacle dicts
        bounds: List [x_min, x_max, y_min, y_max]
        max_iter: Maximum number of iterations
        step_size: Maximum edge length
        goal_bias: Probability of sampling the goal
        goal_threshold: Distance threshold to consider goal reached

    Returns:
        nodes: Shape (N, 2) — tree node positions
        parents: Shape (N,) — parent index for each node (-1 for root)
        goal_idx: Index of the goal node, or -1 if not found
        path_cost_history: List of (iteration, path_cost) when goal is first/better reached
    """
    nodes = [np.array(start, dtype=float)]
    parents = [-1]
    goal = np.array(goal, dtype=float)
    goal_idx = -1
    path_cost_history = []

    for i in range(max_iter):
        # Sample
        if np.random.random() < goal_bias:
            q_rand = goal.copy()
        else:
            q_rand = sample_random(bounds)

        # Nearest neighbor
        tree_array = np.array(nodes)
        near_idx = nearest_neighbor(tree_array, q_rand)
        q_near = nodes[near_idx]

        # Steer
        q_new = steer(q_near, q_rand, step_size)

        # Collision check
        if collision_free(q_near, q_new, obstacles):
            nodes.append(q_new)
            parents.append(near_idx)
            new_idx = len(nodes) - 1

            # Check if goal reached
            if np.linalg.norm(q_new - goal) < goal_threshold:
                if goal_idx == -1:  # First time reaching goal
                    goal_idx = new_idx
                    # Compute path cost
                    cost = _compute_path_cost(nodes, parents, new_idx)
                    path_cost_history.append((i, cost))

    return np.array(nodes), np.array(parents), goal_idx, path_cost_history


def _compute_path_cost(nodes, parents, goal_idx):
    """Compute total Euclidean path cost from start to goal_idx.

    Args:
        nodes: List of node positions
        parents: List of parent indices
        goal_idx: Index of the goal node

    Returns:
        cost: Total Euclidean distance along the path
    """
    cost = 0.0
    idx = goal_idx
    while parents[idx] != -1:
        parent = parents[idx]
        cost += np.linalg.norm(np.array(nodes[idx]) - np.array(nodes[parent]))
        idx = parent
    return cost


def extract_path(nodes, parents, goal_idx):
    """Extract the path from start to goal by backtracking through parents.

    Args:
        nodes: Shape (N, 2) — tree node positions
        parents: Shape (N,) — parent indices
        goal_idx: Index of the goal node

    Returns:
        path: Shape (M, 2) — ordered path from start to goal
    """
    if goal_idx == -1:
        return np.array([])

    path = []
    idx = goal_idx
    while idx != -1:
        path.append(nodes[idx])
        idx = parents[idx]
    return np.array(path[::-1])


# Verify basic functions
test_bounds = [0, 10, 0, 10]
test_sample = sample_random(test_bounds)
print(f"Random sample in bounds: ({test_sample[0]:.2f}, {test_sample[1]:.2f}) "
      f"[{'PASS' if 0 <= test_sample[0] <= 10 and 0 <= test_sample[1] <= 10 else 'FAIL'}]")

test_nodes = np.array([[0, 0], [1, 1], [3, 4]])
test_query = np.array([2, 2])
nn_idx = nearest_neighbor(test_nodes, test_query)
print(f"Nearest neighbor to (2,2): index={nn_idx}, node=({test_nodes[nn_idx][0]},{test_nodes[nn_idx][1]}) "
      f"[{'PASS' if nn_idx == 1 else 'FAIL'}]")

test_steer = steer(np.array([0.0, 0.0]), np.array([10.0, 0.0]), 0.5)
print(f"Steer (0,0)->(10,0) step=0.5: ({test_steer[0]:.2f}, {test_steer[1]:.2f}) "
      f"[{'PASS' if abs(test_steer[0] - 0.5) < 1e-6 else 'FAIL'}]")

---
## 4. RRT in 2D Workspace

We now apply RRT to a point robot navigating in a 2D workspace with both circular and rectangular obstacles. The environment features a **narrow passage** that makes planning challenging — the planner must discover the gap to find a valid path.

In [ ]:
# Define an interesting obstacle environment with narrow passage
workspace_obstacles = [
    # Circular obstacles
    {'center': (2.0, 5.0), 'radius': 1.0},
    {'center': (7.0, 8.0), 'radius': 0.8},
    {'center': (8.0, 3.0), 'radius': 0.7},
    {'center': (4.0, 2.0), 'radius': 0.6},
    # Rectangular obstacles forming a wall with narrow passage
    {'corner': (4.5, 0.0), 'width': 0.4, 'height': 4.0},
    {'corner': (4.5, 5.5), 'width': 0.4, 'height': 4.5},
    # Additional obstacles in open space
    {'center': (6.5, 5.5), 'radius': 0.5},
]

START = np.array([1.0, 1.0])
GOAL = np.array([9.0, 9.0])


def draw_obstacles(ax, obstacles, alpha=0.6):
    """Draw obstacles on a matplotlib axes.

    Args:
        ax: Matplotlib axes object
        obstacles: List of obstacle dicts
        alpha: Transparency value
    """
    for obs in obstacles:
        if 'center' in obs:
            circle = Circle(obs['center'], obs['radius'], color=COLOR_CORAL,
                            alpha=alpha)
            ax.add_patch(circle)
        elif 'corner' in obs:
            rect = Rectangle(obs['corner'], obs['width'], obs['height'],
                             color=COLOR_CORAL, alpha=alpha)
            ax.add_patch(rect)


def draw_tree(ax, nodes, parents, color='gray', alpha=0.3, linewidth=0.5):
    """Draw the RRT tree edges on a matplotlib axes.

    Args:
        ax: Matplotlib axes object
        nodes: Shape (N, 2) — node positions
        parents: Shape (N,) — parent indices
        color: Edge color
        alpha: Edge transparency
        linewidth: Edge width
    """
    lines = []
    for i in range(1, len(nodes)):
        parent = parents[i]
        lines.append([nodes[parent], nodes[i]])
    if lines:
        lc = LineCollection(lines, colors=color, alpha=alpha, linewidths=linewidth)
        ax.add_collection(lc)


# Run RRT
np.random.seed(42)
t0 = time.time()
rrt_nodes, rrt_parents, rrt_goal_idx, rrt_cost_hist = build_rrt(
    START, GOAL, workspace_obstacles, WORKSPACE_BOUNDS,
    max_iter=MAX_ITER, step_size=STEP_SIZE
)
rrt_time = time.time() - t0

rrt_path = extract_path(rrt_nodes, rrt_parents, rrt_goal_idx)

if len(rrt_path) > 0:
    rrt_path_cost = sum(np.linalg.norm(rrt_path[i+1] - rrt_path[i])
                        for i in range(len(rrt_path) - 1))
    print(f"RRT path found: {len(rrt_path)} waypoints, cost = {rrt_path_cost:.2f}")
else:
    rrt_path_cost = float('inf')
    print("RRT: No path found!")

print(f"Tree size: {len(rrt_nodes)} nodes")
print(f"Computation time: {rrt_time:.3f} s")

In [ ]:
# Visualize RRT tree growth at different iteration counts
iter_snapshots = [100, 500, 1000, MAX_ITER]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

np.random.seed(42)

for ax_idx, max_it in enumerate(iter_snapshots):
    np.random.seed(42)
    nodes_snap, parents_snap, goal_snap, _ = build_rrt(
        START, GOAL, workspace_obstacles, WORKSPACE_BOUNDS,
        max_iter=max_it, step_size=STEP_SIZE
    )

    ax = axes[ax_idx]
    title_suffix = '(Final)' if max_it == MAX_ITER else ''
    ax.set_title(f'RRT at {max_it} iterations {title_suffix}',
                 fontsize=13, fontweight='bold')
    ax.set_xlim(-0.5, 10.5)
    ax.set_ylim(-0.5, 10.5)
    ax.set_aspect('equal')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    draw_obstacles(ax, workspace_obstacles)
    draw_tree(ax, nodes_snap, parents_snap, color='gray', alpha=0.3, linewidth=0.5)

    # Draw path if found
    path_snap = extract_path(nodes_snap, parents_snap, goal_snap)
    if len(path_snap) > 0:
        ax.plot(path_snap[:, 0], path_snap[:, 1], '-', color=COLOR_STEELBLUE,
                linewidth=3, zorder=5, label='Path')

    # Start and goal
    ax.plot(*START, 'o', color=COLOR_SEAGREEN, markersize=12,
            markeredgecolor='black', zorder=10, label='Start')
    ax.plot(*GOAL, '*', color=COLOR_GOLDENROD, markersize=15,
            markeredgecolor='black', zorder=10, label='Goal')

    ax.legend(loc='lower right', fontsize=9)
    ax.text(0.02, 0.98, f'Nodes: {len(nodes_snap)}',
            transform=ax.transAxes, fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('RRT Tree Growth Over Iterations', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Probabilistic Completeness

RRT is **probabilistically complete**: if a solution exists, the probability of finding it approaches 1 as the number of samples $n \to \infty$.

### Sketch of Proof

Let the solution path pass through a sequence of "attraction regions" — balls of radius $\epsilon$ centered along a collision-free path. At each iteration:

- The probability of sampling in a useful region is at least $p_{\min} = \frac{\text{Vol}(B_\epsilon)}{\text{Vol}(\mathcal{C})} > 0$
- The probability of *not* sampling in any useful region after $n$ iterations is at most $(1 - p_{\min})^n$

Therefore:

$$\boxed{P(\text{failure after } n \text{ samples}) \leq (1 - p_{\min})^n \to 0 \text{ as } n \to \infty}$$

The failure probability decreases **exponentially** with the number of samples.

In [ ]:
# Numerical verification: Run RRT many times, compute success rate vs max_iter
iter_values = [100, 250, 500, 1000, 1500, 2000, 3000, 5000]
n_trials = 100
success_rates = []

print("Running probabilistic completeness experiment...")
print(f"  {n_trials} trials per iteration count\n")

for max_it in iter_values:
    successes = 0
    for trial in range(n_trials):
        np.random.seed(trial * 1000 + max_it)  # Different seed per trial
        _, _, goal_idx, _ = build_rrt(
            START, GOAL, workspace_obstacles, WORKSPACE_BOUNDS,
            max_iter=max_it, step_size=STEP_SIZE,
            goal_bias=GOAL_BIAS
        )
        if goal_idx != -1:
            successes += 1
    rate = successes / n_trials
    success_rates.append(rate)
    print(f"  max_iter={max_it:5d}: success rate = {rate:.2f}")

# Final verification
rate_5000 = success_rates[-1]
print(f"\nProb completeness: success rate at N=5000 = {rate_5000:.2f} "
      f"[{'PASS' if rate_5000 > 0.95 else 'FAIL'}]")

In [ ]:
# Plot success rate vs iterations
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(iter_values, success_rates, 'o-', color=COLOR_STEELBLUE,
        markersize=8, linewidth=2, label='Empirical success rate')
ax.axhline(y=1.0, color=COLOR_CORAL, linestyle='--', alpha=0.6,
           label='Theoretical limit (1.0)')
ax.axhline(y=0.95, color=COLOR_SEAGREEN, linestyle=':', alpha=0.6,
           label='95% threshold')
ax.set_xlabel('Maximum Iterations', fontsize=13)
ax.set_ylabel('Success Rate', fontsize=13)
ax.set_title('RRT Probabilistic Completeness Verification',
             fontsize=14, fontweight='bold')
ax.set_ylim(-0.05, 1.1)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 6. RRT* — Asymptotic Optimality

While RRT finds *a* path, **RRT*** (RRT-star) finds increasingly **optimal** paths as more samples are added. Two key modifications:

### 1. Choose Best Parent

Instead of connecting $q_{\text{new}}$ to its nearest neighbor, check all nodes within a radius:

$$\boxed{r_n = \gamma \cdot \left(\frac{\log n}{n}\right)^{1/d}}$$

where $d$ is the space dimension, $n$ is the current tree size, and $\gamma > \gamma^* = 2(1 + 1/d)^{1/d} \cdot (\mu(\mathcal{C}_{\text{free}}) / \zeta_d)^{1/d}$ ensures asymptotic optimality.

Among these near neighbors, choose the parent that minimizes the **cost-to-come** $c(q_{\text{new}})$.

### 2. Rewiring

After adding $q_{\text{new}}$, check if it can serve as a better parent for nearby nodes. For each near neighbor $q_{\text{near}}$:

$$\text{If } c(q_{\text{new}}) + \|q_{\text{new}} - q_{\text{near}}\| < c(q_{\text{near}}) \text{ and edge is collision-free, rewire.}$$

### Comparison

| Property | RRT | RRT* |
|----------|-----|------|
| Completeness | Probabilistic | Probabilistic |
| Optimality | No guarantee | Asymptotically optimal |
| Cost per iteration | $O(\log n)$ | $O(n)$ (rewiring) |
| Path quality | Jagged, suboptimal | Converges to optimal |

In [ ]:
def build_rrt_star(start, goal, obstacles, bounds, max_iter=MAX_ITER,
                   step_size=STEP_SIZE, gamma=GAMMA_RRT_STAR,
                   goal_bias=GOAL_BIAS, goal_threshold=GOAL_THRESHOLD):
    """Build an RRT* tree from start toward goal with rewiring.

    RRT* extends RRT with two modifications:
    1. Choose best parent among near neighbors (minimize cost-to-come)
    2. Rewire near neighbors through new node if it reduces their cost

    Args:
        start: Shape (2,) — start position
        goal: Shape (2,) — goal position
        obstacles: List of obstacle dicts
        bounds: List [x_min, x_max, y_min, y_max]
        max_iter: Maximum number of iterations
        step_size: Maximum edge length
        gamma: RRT* rewiring radius constant
        goal_bias: Probability of sampling the goal
        goal_threshold: Distance to consider goal reached

    Returns:
        nodes: Shape (N, 2) — tree node positions
        parents: Shape (N,) — parent index for each node
        costs: Shape (N,) — cost-to-come for each node
        goal_idx: Index of the best goal node, or -1
        path_cost_history: List of (iteration, best_path_cost) over time
        rewire_count: Total number of rewiring operations performed
    """
    d = 2  # Dimension
    nodes = [np.array(start, dtype=float)]
    parents = [-1]
    costs = [0.0]  # Cost-to-come
    goal = np.array(goal, dtype=float)
    goal_idx = -1
    best_goal_cost = float('inf')
    path_cost_history = []
    rewire_count = 0

    for i in range(max_iter):
        # Sample
        if np.random.random() < goal_bias:
            q_rand = goal.copy()
        else:
            q_rand = sample_random(bounds)

        # Nearest neighbor
        tree_array = np.array(nodes)
        near_idx = nearest_neighbor(tree_array, q_rand)
        q_near = nodes[near_idx]

        # Steer
        q_new = steer(q_near, q_rand, step_size)

        # Collision check
        if not collision_free(q_near, q_new, obstacles):
            continue

        # Compute rewiring radius
        n = len(nodes)
        r_n = min(gamma * (np.log(n + 1) / (n + 1)) ** (1.0 / d), step_size * 3)

        # Find near neighbors within radius r_n
        dists = np.linalg.norm(tree_array - q_new, axis=1)
        near_indices = np.where(dists < r_n)[0]

        # Choose best parent (minimize cost-to-come)
        best_parent = near_idx
        best_cost = costs[near_idx] + np.linalg.norm(q_new - q_near)

        for j in near_indices:
            candidate_cost = costs[j] + np.linalg.norm(q_new - nodes[j])
            if candidate_cost < best_cost:
                if collision_free(nodes[j], q_new, obstacles):
                    best_parent = j
                    best_cost = candidate_cost

        # Add new node
        nodes.append(q_new)
        parents.append(best_parent)
        costs.append(best_cost)
        new_idx = len(nodes) - 1

        # Rewire: check if new node provides better path to nearby nodes
        for j in near_indices:
            new_cost_j = best_cost + np.linalg.norm(q_new - nodes[j])
            if new_cost_j < costs[j]:
                if collision_free(q_new, nodes[j], obstacles):
                    parents[j] = new_idx
                    # Propagate cost improvement to subtree
                    old_cost_j = costs[j]
                    costs[j] = new_cost_j
                    rewire_count += 1
                    # Propagate to children
                    _propagate_cost_improvement(j, old_cost_j - new_cost_j,
                                               nodes, parents, costs)

        # Check if goal reached
        if np.linalg.norm(q_new - goal) < goal_threshold:
            if best_cost < best_goal_cost:
                goal_idx = new_idx
                best_goal_cost = best_cost
                path_cost_history.append((i, best_cost))

        # Record periodic cost updates
        if goal_idx != -1 and i % 100 == 0:
            current_cost = costs[goal_idx]
            if not path_cost_history or current_cost < path_cost_history[-1][1]:
                path_cost_history.append((i, current_cost))

    return (np.array(nodes), np.array(parents), np.array(costs),
            goal_idx, path_cost_history, rewire_count)


def _propagate_cost_improvement(node_idx, improvement, nodes, parents, costs):
    """Propagate a cost improvement through the subtree rooted at node_idx.

    Args:
        node_idx: Index of the node whose cost was reduced
        improvement: Positive value representing cost reduction
        nodes: List of node positions
        parents: List of parent indices
        costs: List of cost-to-come values (modified in place)
    """
    stack = []
    for j in range(len(nodes)):
        if parents[j] == node_idx:
            stack.append(j)

    while stack:
        idx = stack.pop()
        costs[idx] -= improvement
        for j in range(len(nodes)):
            if parents[j] == idx:
                stack.append(j)


# Run RRT*
np.random.seed(42)
t0 = time.time()
rrt_star_nodes, rrt_star_parents, rrt_star_costs, rrt_star_goal_idx, \
    rrt_star_cost_hist, rrt_star_rewires = build_rrt_star(
        START, GOAL, workspace_obstacles, WORKSPACE_BOUNDS,
        max_iter=MAX_ITER, step_size=STEP_SIZE
    )
rrt_star_time = time.time() - t0

rrt_star_path = extract_path(rrt_star_nodes, rrt_star_parents, rrt_star_goal_idx)

if len(rrt_star_path) > 0:
    rrt_star_path_cost = sum(np.linalg.norm(rrt_star_path[i+1] - rrt_star_path[i])
                             for i in range(len(rrt_star_path) - 1))
    print(f"RRT* path found: {len(rrt_star_path)} waypoints, cost = {rrt_star_path_cost:.2f}")
else:
    rrt_star_path_cost = float('inf')
    print("RRT*: No path found!")

print(f"Tree size: {len(rrt_star_nodes)} nodes")
print(f"Rewiring operations: {rrt_star_rewires}")
print(f"Computation time: {rrt_star_time:.3f} s")

---
## 7. RRT vs RRT* Comparison

We compare RRT and RRT* on the same environment with the same random seed to isolate the effect of the rewiring step.

In [ ]:
# Run both algorithms with multiple iteration counts to track cost convergence
iter_checkpoints = [500, 1000, 1500, 2000, 2500, 3000, 4000, 5000]
rrt_costs_over_iter = []
rrt_star_costs_over_iter = []

print("Running RRT vs RRT* comparison...\n")

for max_it in iter_checkpoints:
    # RRT
    np.random.seed(42)
    nodes_r, parents_r, gidx_r, _ = build_rrt(
        START, GOAL, workspace_obstacles, WORKSPACE_BOUNDS,
        max_iter=max_it, step_size=STEP_SIZE
    )
    path_r = extract_path(nodes_r, parents_r, gidx_r)
    if len(path_r) > 0:
        cost_r = sum(np.linalg.norm(path_r[i+1] - path_r[i])
                     for i in range(len(path_r) - 1))
    else:
        cost_r = float('inf')
    rrt_costs_over_iter.append(cost_r)

    # RRT*
    np.random.seed(42)
    nodes_rs, parents_rs, costs_rs, gidx_rs, _, _ = build_rrt_star(
        START, GOAL, workspace_obstacles, WORKSPACE_BOUNDS,
        max_iter=max_it, step_size=STEP_SIZE
    )
    path_rs = extract_path(nodes_rs, parents_rs, gidx_rs)
    if len(path_rs) > 0:
        cost_rs = sum(np.linalg.norm(path_rs[i+1] - path_rs[i])
                      for i in range(len(path_rs) - 1))
    else:
        cost_rs = float('inf')
    rrt_star_costs_over_iter.append(cost_rs)

    print(f"  iter={max_it:5d}: RRT cost = {cost_r:7.2f}, RRT* cost = {cost_rs:7.2f}")

# Final comparison
final_rrt = rrt_costs_over_iter[-1]
final_rrt_star = rrt_star_costs_over_iter[-1]
if final_rrt < float('inf') and final_rrt_star < float('inf'):
    improvement = (final_rrt - final_rrt_star) / final_rrt * 100
    print(f"\nRRT path cost: {final_rrt:.2f}, RRT* path cost: {final_rrt_star:.2f}, "
          f"improvement: {improvement:.1f}%")
else:
    print(f"\nRRT path cost: {final_rrt:.2f}, RRT* path cost: {final_rrt_star:.2f}")

In [ ]:
# Side-by-side visualization and cost convergence
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Panel 1: RRT tree + path ---
ax = axes[0]
ax.set_title('RRT', fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 10.5)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')

draw_obstacles(ax, workspace_obstacles)
draw_tree(ax, rrt_nodes, rrt_parents, color='gray', alpha=0.2, linewidth=0.4)
if len(rrt_path) > 0:
    ax.plot(rrt_path[:, 0], rrt_path[:, 1], '-', color=COLOR_STEELBLUE,
            linewidth=3, zorder=5, label=f'Path (cost={rrt_path_cost:.1f})')
ax.plot(*START, 'o', color=COLOR_SEAGREEN, markersize=12,
        markeredgecolor='black', zorder=10, label='Start')
ax.plot(*GOAL, '*', color=COLOR_GOLDENROD, markersize=15,
        markeredgecolor='black', zorder=10, label='Goal')
ax.legend(loc='lower right', fontsize=9)

# --- Panel 2: RRT* tree + path ---
ax = axes[1]
ax.set_title('RRT*', fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 10.5)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')

draw_obstacles(ax, workspace_obstacles)
draw_tree(ax, rrt_star_nodes, rrt_star_parents, color='gray', alpha=0.2, linewidth=0.4)
if len(rrt_star_path) > 0:
    ax.plot(rrt_star_path[:, 0], rrt_star_path[:, 1], '-', color=COLOR_CORAL,
            linewidth=3, zorder=5, label=f'Path (cost={rrt_star_path_cost:.1f})')
ax.plot(*START, 'o', color=COLOR_SEAGREEN, markersize=12,
        markeredgecolor='black', zorder=10, label='Start')
ax.plot(*GOAL, '*', color=COLOR_GOLDENROD, markersize=15,
        markeredgecolor='black', zorder=10, label='Goal')
ax.legend(loc='lower right', fontsize=9)

# --- Panel 3: Cost convergence ---
ax = axes[2]
ax.set_title('Path Cost vs Iterations', fontsize=14, fontweight='bold')

# Filter out inf values for plotting
valid_rrt = [(it, c) for it, c in zip(iter_checkpoints, rrt_costs_over_iter)
             if c < float('inf')]
valid_rrt_star = [(it, c) for it, c in zip(iter_checkpoints, rrt_star_costs_over_iter)
                  if c < float('inf')]

if valid_rrt:
    iters_r, costs_r = zip(*valid_rrt)
    ax.plot(iters_r, costs_r, 'o-', color=COLOR_STEELBLUE, markersize=7,
            label='RRT')
if valid_rrt_star:
    iters_rs, costs_rs = zip(*valid_rrt_star)
    ax.plot(iters_rs, costs_rs, 's-', color=COLOR_CORAL, markersize=7,
            label='RRT*')

ax.set_xlabel('Iterations', fontsize=13)
ax.set_ylabel('Path Cost (Euclidean)', fontsize=13)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

---
## 8. C-Space Planning for 3R Arm

We now tackle a higher-dimensional problem: planning for a **3R planar arm** with link lengths $L_1 = 1.0$, $L_2 = 0.8$, $L_3 = 0.5$. The configuration space is 3-dimensional: $\mathcal{C} = [0, 2\pi)^3$.

Key adaptations for C-space planning:

- **Distance metric**: Use angular distance with wrapping: $d(\theta_a, \theta_b) = \sqrt{\sum_i \min(|\theta_{a,i} - \theta_{b,i}|, 2\pi - |\theta_{a,i} - \theta_{b,i}|)^2}$
- **Collision checking**: Evaluate forward kinematics at sampled configurations along the edge
- **Steering**: Move in joint space, wrapping angles to $[0, 2\pi)$

This demonstrates the power of sampling-based planning: we never explicitly construct the C-space obstacle map (which would require an expensive 3D grid), but instead check collisions on-the-fly.

For a comparison with optimization-based approaches (e.g., CHOMP), see the companion CHOMP notebook.

In [ ]:
def arm_3r_forward_kinematics(q1, q2, q3, l1=L1_3R, l2=L2_3R, l3=L3_3R):
    """Compute joint positions for a 3R planar arm.

    Args:
        q1: Angle of joint 1 (radians)
        q2: Angle of joint 2 (radians), relative to link 1
        q3: Angle of joint 3 (radians), relative to link 2
        l1, l2, l3: Link lengths

    Returns:
        positions: Shape (4, 2) with [base, joint1, joint2, end-effector]
    """
    base = np.array([0.0, 0.0])
    joint1 = base + np.array([l1 * np.cos(q1), l1 * np.sin(q1)])
    joint2 = joint1 + np.array([l2 * np.cos(q1 + q2), l2 * np.sin(q1 + q2)])
    end_eff = joint2 + np.array([
        l3 * np.cos(q1 + q2 + q3), l3 * np.sin(q1 + q2 + q3)
    ])
    return np.array([base, joint1, joint2, end_eff])


def arm_3r_collision(q1, q2, q3, obstacles, l1=L1_3R, l2=L2_3R, l3=L3_3R):
    """Check if a 3R arm configuration collides with circular obstacles.

    Tests all three links against all obstacles.

    Args:
        q1, q2, q3: Joint angles (radians)
        obstacles: List of dicts with 'center' and 'radius' keys
        l1, l2, l3: Link lengths

    Returns:
        collision: True if any link collides with any obstacle
    """
    positions = arm_3r_forward_kinematics(q1, q2, q3, l1, l2, l3)
    # Check each link
    for link_idx in range(3):
        seg_start = positions[link_idx]
        seg_end = positions[link_idx + 1]
        for obs in obstacles:
            center = np.array(obs['center'])
            radius = obs['radius']
            if point_to_segment_distance(center, seg_start, seg_end) < radius:
                return True
    return False


def angular_distance(q1, q2):
    """Compute angular distance between two joint-space configurations.

    Uses the minimum of direct and wrapped difference for each dimension.

    Args:
        q1: Shape (d,) — first configuration
        q2: Shape (d,) — second configuration

    Returns:
        dist: Euclidean norm of element-wise angular differences
    """
    diff = np.abs(q1 - q2)
    diff = np.minimum(diff, 2 * np.pi - diff)
    return np.linalg.norm(diff)


def wrap_angle(theta):
    """Wrap angle to [0, 2*pi).

    Args:
        theta: Angle or array of angles (radians)

    Returns:
        wrapped: Angle(s) in [0, 2*pi)
    """
    return theta % (2 * np.pi)


def steer_angular(from_q, to_q, step_size):
    """Steer in joint space with angular wrapping.

    Computes the shortest angular direction for each joint and moves
    by at most step_size.

    Args:
        from_q: Shape (d,) — start configuration
        to_q: Shape (d,) — target configuration
        step_size: Maximum step size in joint space

    Returns:
        new_q: Shape (d,) — new configuration, angles wrapped to [0, 2*pi)
    """
    diff = to_q - from_q
    # Wrap differences to [-pi, pi]
    diff = (diff + np.pi) % (2 * np.pi) - np.pi
    dist = np.linalg.norm(diff)
    if dist < step_size:
        return wrap_angle(to_q.copy())
    new_q = from_q + (diff / dist) * step_size
    return wrap_angle(new_q)


def build_rrt_star_3r(start_q, goal_q, obstacles, max_iter=2000,
                      step_size=0.3, gamma=GAMMA_3D,
                      goal_bias=0.1, goal_threshold=0.3):
    """Build an RRT* tree in 3D joint space for a 3R arm.

    Uses angular distance metric and collision checking via forward kinematics.

    Args:
        start_q: Shape (3,) — start joint configuration
        goal_q: Shape (3,) — goal joint configuration
        obstacles: List of circular obstacle dicts (in workspace)
        max_iter: Maximum iterations
        step_size: Maximum step in joint space
        gamma: RRT* radius parameter
        goal_bias: Probability of sampling goal
        goal_threshold: Angular distance threshold for goal

    Returns:
        nodes: Shape (N, 3) — joint-space node positions
        parents: Shape (N,) — parent indices
        costs: Shape (N,) — cost-to-come
        goal_idx: Best goal node index, or -1
    """
    d = 3  # Dimension
    nodes = [np.array(start_q, dtype=float)]
    parents = [-1]
    costs = [0.0]
    goal_q = np.array(goal_q, dtype=float)
    goal_idx = -1
    best_goal_cost = float('inf')

    def is_edge_free(q_a, q_b, n_checks=15):
        """Check if edge from q_a to q_b is collision-free in joint space."""
        for t in np.linspace(0, 1, n_checks):
            q_interp = q_a + t * ((q_b - q_a + np.pi) % (2 * np.pi) - np.pi)
            q_interp = wrap_angle(q_interp)
            if arm_3r_collision(q_interp[0], q_interp[1], q_interp[2], obstacles):
                return False
        return True

    for i in range(max_iter):
        # Sample
        if np.random.random() < goal_bias:
            q_rand = goal_q.copy()
        else:
            q_rand = np.random.uniform(0, 2 * np.pi, size=3)

        # Nearest neighbor (angular distance)
        tree_array = np.array(nodes)
        dists = np.array([angular_distance(tree_array[j], q_rand)
                          for j in range(len(tree_array))])
        near_idx = int(np.argmin(dists))
        q_near = nodes[near_idx]

        # Steer
        q_new = steer_angular(q_near, q_rand, step_size)

        # Collision check
        if not is_edge_free(q_near, q_new):
            continue

        # Near neighbors for RRT*
        n = len(nodes)
        r_n = min(gamma * (np.log(n + 1) / (n + 1)) ** (1.0 / d), step_size * 3)
        near_dists = np.array([angular_distance(tree_array[j], q_new)
                               for j in range(len(tree_array))])
        near_indices = np.where(near_dists < r_n)[0]

        # Choose best parent
        best_parent = near_idx
        best_cost = costs[near_idx] + angular_distance(q_near, q_new)

        for j in near_indices:
            candidate_cost = costs[j] + angular_distance(nodes[j], q_new)
            if candidate_cost < best_cost:
                if is_edge_free(nodes[j], q_new):
                    best_parent = j
                    best_cost = candidate_cost

        # Add node
        nodes.append(q_new)
        parents.append(best_parent)
        costs.append(best_cost)
        new_idx = len(nodes) - 1

        # Rewire
        for j in near_indices:
            new_cost_j = best_cost + angular_distance(q_new, nodes[j])
            if new_cost_j < costs[j]:
                if is_edge_free(q_new, nodes[j]):
                    parents[j] = new_idx
                    costs[j] = new_cost_j

        # Check goal
        if angular_distance(q_new, goal_q) < goal_threshold:
            if best_cost < best_goal_cost:
                goal_idx = new_idx
                best_goal_cost = best_cost

    return np.array(nodes), np.array(parents), np.array(costs), goal_idx


# Define workspace obstacles for the 3R arm
arm_3r_obstacles = [
    {'center': (1.2, 0.5), 'radius': 0.25},
    {'center': (0.0, 1.5), 'radius': 0.3},
    {'center': (-1.0, 0.8), 'radius': 0.2},
    {'center': (0.7, -0.8), 'radius': 0.25},
]

# Start and goal configurations
q_start_3r = np.array([0.5, 0.8, 0.3])
q_goal_3r = np.array([2.5, 1.5, 1.0])

# Verify start and goal are collision-free
start_free = not arm_3r_collision(*q_start_3r, arm_3r_obstacles)
goal_free = not arm_3r_collision(*q_goal_3r, arm_3r_obstacles)
print(f"Start configuration collision-free: {start_free} [{'PASS' if start_free else 'FAIL'}]")
print(f"Goal configuration collision-free: {goal_free} [{'PASS' if goal_free else 'FAIL'}]")

# Run RRT* in 3D joint space
np.random.seed(42)
print("\nRunning RRT* in 3D joint space...")
t0 = time.time()
nodes_3r, parents_3r, costs_3r, goal_idx_3r = build_rrt_star_3r(
    q_start_3r, q_goal_3r, arm_3r_obstacles,
    max_iter=3000, step_size=0.3
)
time_3r = time.time() - t0

path_3r = extract_path(nodes_3r, parents_3r, goal_idx_3r)

if len(path_3r) > 0:
    path_cost_3r = sum(angular_distance(path_3r[i], path_3r[i+1])
                       for i in range(len(path_3r) - 1))
    print(f"3R arm path found: {len(path_3r)} waypoints, cost = {path_cost_3r:.2f}")
else:
    print("3R arm: No path found!")

print(f"Tree size: {len(nodes_3r)} nodes")
print(f"Computation time: {time_3r:.3f} s")

In [ ]:
# Visualize 3R arm planning results
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Panel 1: C-space tree projected onto q1-q2 plane ---
ax = axes[0]
ax.set_title('C-Space Tree (projected onto $q_1$-$q_2$)',
             fontsize=14, fontweight='bold')

# Draw tree edges
for i in range(1, len(nodes_3r)):
    parent = parents_3r[i]
    ax.plot([nodes_3r[parent, 0], nodes_3r[i, 0]],
            [nodes_3r[parent, 1], nodes_3r[i, 1]],
            '-', color='gray', alpha=0.15, linewidth=0.3)

# Draw path in C-space
if len(path_3r) > 0:
    ax.plot(path_3r[:, 0], path_3r[:, 1], 'o-', color=COLOR_STEELBLUE,
            linewidth=3, markersize=4, zorder=5, label='Path')

ax.plot(q_start_3r[0], q_start_3r[1], 'o', color=COLOR_SEAGREEN,
        markersize=14, markeredgecolor='black', zorder=10, label='Start')
ax.plot(q_goal_3r[0], q_goal_3r[1], '*', color=COLOR_GOLDENROD,
        markersize=16, markeredgecolor='black', zorder=10, label='Goal')

ax.set_xlabel(r'$q_1$ (rad)', fontsize=13)
ax.set_ylabel(r'$q_2$ (rad)', fontsize=13)
ax.set_xlim(0, 2 * np.pi)
ax.set_ylim(0, 2 * np.pi)
ax.legend(fontsize=11)

# --- Panel 2: Workspace trajectory ---
ax = axes[1]
ax.set_title('Workspace: 3R Arm Trajectory', fontsize=14, fontweight='bold')
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)
ax.set_aspect('equal')
ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('y', fontsize=13)

# Draw obstacles
for obs in arm_3r_obstacles:
    circle = Circle(obs['center'], obs['radius'], color=COLOR_CORAL, alpha=0.6)
    ax.add_patch(circle)

# Draw reach boundary
r_max = L1_3R + L2_3R + L3_3R
theta_vis = np.linspace(0, 2 * np.pi, 200)
ax.plot(r_max * np.cos(theta_vis), r_max * np.sin(theta_vis),
        '--', color='gray', alpha=0.3)

# Draw arm at several configurations along the path
if len(path_3r) > 0:
    n_configs = min(8, len(path_3r))
    config_indices = np.linspace(0, len(path_3r) - 1, n_configs, dtype=int)
    cmap = plt.cm.viridis

    for k, ci in enumerate(config_indices):
        q = path_3r[ci]
        positions = arm_3r_forward_kinematics(q[0], q[1], q[2])
        color = cmap(k / (n_configs - 1)) if n_configs > 1 else cmap(0.5)
        alpha = 0.3 + 0.7 * (k / max(n_configs - 1, 1))
        ax.plot(positions[:, 0], positions[:, 1], 'o-', color=color,
                linewidth=3, markersize=5, alpha=alpha)

    # Draw end-effector trace
    ee_trace = np.array([arm_3r_forward_kinematics(q[0], q[1], q[2])[-1]
                         for q in path_3r])
    ax.plot(ee_trace[:, 0], ee_trace[:, 1], '--', color=COLOR_STEELBLUE,
            linewidth=1.5, alpha=0.7, label='End-effector trace')

ax.legend(fontsize=10)
ax.text(0.02, 0.02, 'See CHOMP notebook\nfor comparison',
        transform=ax.transAxes, fontsize=9, style='italic',
        verticalalignment='bottom',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

---
## 9. Summary Visualizations

A comprehensive 4-panel summary of the key results from this notebook.

In [ ]:
# Summary 4-panel figure
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# ---- Panel 1: RRT tree growth (final state) ----
ax = axes[0, 0]
ax.set_title('(a) RRT Final Tree and Path', fontsize=13, fontweight='bold')
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 10.5)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')

draw_obstacles(ax, workspace_obstacles)
draw_tree(ax, rrt_nodes, rrt_parents, color='gray', alpha=0.2, linewidth=0.3)
if len(rrt_path) > 0:
    ax.plot(rrt_path[:, 0], rrt_path[:, 1], '-', color=COLOR_STEELBLUE,
            linewidth=3, zorder=5, label='RRT path')
ax.plot(*START, 'o', color=COLOR_SEAGREEN, markersize=10,
        markeredgecolor='black', zorder=10)
ax.plot(*GOAL, '*', color=COLOR_GOLDENROD, markersize=13,
        markeredgecolor='black', zorder=10)
ax.legend(fontsize=10, loc='lower right')

# ---- Panel 2: RRT* tree with rewired connections highlighted ----
ax = axes[0, 1]
ax.set_title('(b) RRT* Tree with Rewired Connections', fontsize=13, fontweight='bold')
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 10.5)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')

draw_obstacles(ax, workspace_obstacles)

# Draw all tree edges, highlighting those on the optimal path
path_set = set()
if len(rrt_star_path) > 0:
    idx = rrt_star_goal_idx
    while rrt_star_parents[idx] != -1:
        path_set.add((rrt_star_parents[idx], idx))
        idx = rrt_star_parents[idx]

# Regular edges
regular_lines = []
path_lines = []
for i in range(1, len(rrt_star_nodes)):
    parent = rrt_star_parents[i]
    if (parent, i) in path_set:
        path_lines.append([rrt_star_nodes[parent], rrt_star_nodes[i]])
    else:
        regular_lines.append([rrt_star_nodes[parent], rrt_star_nodes[i]])

if regular_lines:
    lc_reg = LineCollection(regular_lines, colors='gray', alpha=0.15, linewidths=0.3)
    ax.add_collection(lc_reg)
if path_lines:
    lc_path = LineCollection(path_lines, colors=COLOR_CORAL, alpha=0.9, linewidths=3)
    ax.add_collection(lc_path)

ax.plot(*START, 'o', color=COLOR_SEAGREEN, markersize=10,
        markeredgecolor='black', zorder=10)
ax.plot(*GOAL, '*', color=COLOR_GOLDENROD, markersize=13,
        markeredgecolor='black', zorder=10)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', linewidth=1, alpha=0.5, label='Tree edges'),
    Line2D([0], [0], color=COLOR_CORAL, linewidth=3, label='Optimal path'),
]
ax.legend(handles=legend_elements, fontsize=10, loc='lower right')

# ---- Panel 3: Path cost convergence ----
ax = axes[1, 0]
ax.set_title('(c) Path Cost Convergence: RRT vs RRT*', fontsize=13, fontweight='bold')

if valid_rrt:
    iters_r, costs_r = zip(*valid_rrt)
    ax.plot(iters_r, costs_r, 'o-', color=COLOR_STEELBLUE, markersize=7,
            label='RRT', linewidth=2)
if valid_rrt_star:
    iters_rs, costs_rs = zip(*valid_rrt_star)
    ax.plot(iters_rs, costs_rs, 's-', color=COLOR_CORAL, markersize=7,
            label='RRT*', linewidth=2)

# Straight-line lower bound
straight_line = np.linalg.norm(GOAL - START)
ax.axhline(y=straight_line, color=COLOR_SEAGREEN, linestyle=':',
           alpha=0.6, label=f'Straight-line distance ({straight_line:.1f})')

ax.set_xlabel('Iterations', fontsize=13)
ax.set_ylabel('Path Cost', fontsize=13)
ax.legend(fontsize=10)

# ---- Panel 4: C-space plan with workspace arm ----
ax = axes[1, 1]
ax.set_title('(d) 3R Arm: C-Space Plan in Workspace', fontsize=13, fontweight='bold')
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')

# Draw obstacles
for obs in arm_3r_obstacles:
    circle = Circle(obs['center'], obs['radius'], color=COLOR_CORAL, alpha=0.5)
    ax.add_patch(circle)

# Draw arm configurations along path
if len(path_3r) > 0:
    n_show = min(6, len(path_3r))
    show_indices = np.linspace(0, len(path_3r) - 1, n_show, dtype=int)
    cmap = plt.cm.cool

    for k, si in enumerate(show_indices):
        q = path_3r[si]
        positions = arm_3r_forward_kinematics(q[0], q[1], q[2])
        color = cmap(k / max(n_show - 1, 1))
        alpha = 0.4 + 0.6 * (k / max(n_show - 1, 1))
        ax.plot(positions[:, 0], positions[:, 1], 'o-', color=color,
                linewidth=2.5, markersize=4, alpha=alpha)

    # End-effector trace
    ee_trace = np.array([arm_3r_forward_kinematics(q[0], q[1], q[2])[-1]
                         for q in path_3r])
    ax.plot(ee_trace[:, 0], ee_trace[:, 1], '--', color=COLOR_GOLDENROD,
            linewidth=1.5, alpha=0.8, label='End-effector trace')

ax.legend(fontsize=10, loc='lower left')

plt.suptitle('RRT & RRT* Path Planning — Summary',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 10. Extensions and Further Reading

### Bidirectional RRT

Grow two trees simultaneously — one from the start and one from the goal — and attempt to connect them. This dramatically improves performance in environments with narrow passages, as the two trees can "meet in the middle."

- **RRT-Connect** (Kuffner & LaValle, 2000): A greedy variant that extends toward the other tree until blocked.
- Reduces expected planning time from $O(1/p)$ to $O(1/\sqrt{p})$ where $p$ is the probability of sampling in the narrow passage.

### Informed RRT*

Once an initial path of cost $c_{\text{best}}$ is found, future samples are drawn from an **ellipsoidal region** defined by:

$$\mathcal{X}_{\text{focus}} = \{x \in \mathcal{C} \mid \|x - x_{\text{start}}\| + \|x - x_{\text{goal}}\| \leq c_{\text{best}}\}$$

This focuses sampling on regions that could yield shorter paths, significantly accelerating convergence (Gammell et al., 2014).

### Kinodynamic Planning

The basic RRT assumes a **holonomic** robot that can move in any direction. For systems with **dynamics constraints** (e.g., car-like robots with minimum turning radius, quadrotors with acceleration limits), the `steer` function must integrate the system dynamics:

$$\dot{x} = f(x, u), \quad u \in \mathcal{U}$$

Kinodynamic RRT replaces the geometric steering with forward simulation of dynamics under sampled controls.

### Comparison with CHOMP

| Aspect | Sampling-Based (RRT/RRT*) | Optimization-Based (CHOMP) |
|--------|---------------------------|----------------------------|
| Approach | Explore C-space randomly | Optimize a trajectory functional |
| Completeness | Probabilistically complete | Local minima possible |
| Optimality | RRT*: asymptotically optimal | Local optimum only |
| Narrow passages | Can find them (with enough samples) | May get stuck |
| Smoothness | Paths are jagged (need smoothing) | Naturally smooth |
| Speed | Slower in low-$d$ | Fast when initial guess is good |
| High dimensions | Scales well | Gradient computation costly |

In practice, hybrid approaches often work best: use RRT to find an initial feasible path, then refine it with trajectory optimization (CHOMP, TrajOpt, etc.).

### Connection to Potential Fields

**Artificial potential fields** provide a reactive (local) approach to planning:
- Attractive potential toward the goal: $U_{\text{att}}(q) = \frac{1}{2} k \|q - q_{\text{goal}}\|^2$
- Repulsive potential from obstacles: $U_{\text{rep}}(q) \propto \frac{1}{d(q, \mathcal{O})^2}$

Potential fields are fast but suffer from **local minima** (e.g., U-shaped obstacles). RRT avoids local minima through randomized exploration, making it more suitable for complex environments. However, potential fields can be used as **heuristics** to bias the RRT sampling distribution toward promising regions.